In [ ]:
import pandas as pd

In [ ]:
dados = pd.read_csv("../data/WineQT.csv")

O dataset possuí 13 coluns e 1143 linhas.

In [ ]:
dados.shape

In [ ]:
dados.head()

Definindo Variável Target:
Coluna "quality"

In [ ]:
set(dados.quality)

**Analisando os dados do data set**

In [ ]:
dados.describe()

Valores nulos:
Não existe valor nulo no data set

In [ ]:
import missingno as msno
msno.matrix(dados)

In [ ]:
import seaborn as sb

Ao analisar a variavel "quality" percebemos outliers no data set:
O boxplot mostra que a maior parte das avaliações de qualidade está entre as notas 5 e 6, enquanto as notas 3 e 8 aparecem como outliers.


In [ ]:
sb.boxplot(x=dados["quality"])

In [ ]:
sb.histplot(data=dados, x="quality")

A maior parte dos valores de pH está entre aproximadamente 3,2 e 3,4, o que indica uma distribuição centralizada nessa faixa. O boxplot mostra alguns valores extremos abaixo de 2,9 e acima de 3,7, que são os outliers.

In [ ]:
sb.boxplot(x=dados["pH"])

In [ ]:
sb.histplot(data=dados, x="pH")

In [ ]:
sb.boxplot(x=dados["alcohol"])

In [ ]:
sb.histplot(data=dados, x="alcohol")

In [ ]:
sb.boxplot(x=dados["density"])

In [ ]:
sb.histplot(data=dados, x="density")

In [ ]:
sb.boxplot(x=dados["sulphates"])

In [ ]:
sb.histplot(data=dados, x="sulphates")

O Alcool influencia na qualidade do vinho?

Vinhos com qualidade 7 e 8 aparecem com maior frequência em níveis elevados de álcool, e vinhos com qualidade menores estão mais concentrados em níveis mais baixos. Mas acreditamos que o alcool não determina sozinho a qualidade do vinho.

In [ ]:
sb.set_theme(style="whitegrid", palette="muted")

ax = sb.swarmplot(data=dados, x="quality", y="alcohol", hue="quality")
ax.set(ylabel="alcohol")

Média de álcool por qualidade:

Os vinhos com qualidade 7 e 8 apresentam as maiores médias de álcool, isso aponta uma relação positiva entre o teor alcoólico e a qualidade do vinho.

In [ ]:
sb.barplot(data=dados, x="quality", y="alcohol", errorbar=None)

In [ ]:
dados["classificacao"] = dados["quality"].apply(lambda nota: 1 if nota >= 7 else 0)

In [ ]:
dados[["quality", "classificacao"]].drop_duplicates().sort_values("quality")

In [ ]:
# Dependência declarada em requirements.txt

In [ ]:
import plotly.express as px

O gráfico de violino mostra que os vinhos classificados como de alta qualidade apresentam uma concentração de alcool maior do que os vinhos de baixa ou média qualidade.
Enquanto a classe de baixa/média qualidade está mais concentrada próxima de 9,5% a 10% de álcool, a classe de alta qualidade está mais concetrada entre 11% e 12%.


In [ ]:
px.violin(dados,y="alcohol", x="classificacao", color="classificacao", box=True, points="all")

O grafico abaixo traz visualmente a relação entre álcool, densidade, sulfatos, pH e qualidade.
A relação mais evidente ocorre entre álcool e densidade, e indica que vinhos com maior teor alcoólico tendem a apresentar menor densidade.
Os vinhos com qualidade mais altas aparecem com maior frequência em valores elevados de álcool e menores valores de densidade.
Os sulfatos apresentam uma possível relação com a qualidade, mas existe muita sobreposição entre as notas.
O pH não demonstrou uma separação clara entre os diferentes níveis de qualidade.


In [ ]:
sb.pairplot(dados,vars=['alcohol','density','sulphates','pH'],hue="quality")

In [ ]:
import matplotlib.pyplot as plt


Entendendo as correlações:

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
dados.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12,12))
matriz_correlacao = dados.corr(numeric_only=True)
sb.heatmap(data=matriz_correlacao, annot=True, linewidths=.5, ax=ax)

Definição de variaveis:

In [ ]:
x = dados[['alcohol','citric acid', 'pH', 'density', 'sulphates']]
y = dados['classificacao']

In [ ]:
from sklearn.model_selection import train_test_split #separação em treino e teste
from sklearn.neighbors import KNeighborsClassifier   #knn

Separando  base entre treino e teste, mantendo a proporção das classes nos conjuntos de treino e teste e permitindo obter os mesmos resultados em diferentes execuções.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
x_train.shape

In [ ]:
x_test.shape

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

Colocando as variavies em escalas comparaveis

In [ ]:
scaler = StandardScaler()
#scaler = MinMaxScaler()

scaler.fit(x_train)

x_train_escalonado = scaler.transform(x_train)
x_test_escalonado = scaler.transform(x_test)

In [ ]:
import numpy as np

Teste de diferentes valores para a variavel "K" para descobrir qual deles gera menos erros.

In [ ]:
error = []

for i in range(1, 10):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(x_train_escalonado, y_train)
    pred_i = knn.predict(x_test_escalonado)
    error.append(np.mean(pred_i != y_test))

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(range(1, 10), error, color='red', linestyle='dashed', marker='o',
         markerfacecolor='blue', markersize=10)
plt.title('Error Rate K Value')
plt.xlabel('K Value')
plt.ylabel('Mean Error')

In [ ]:
#!pip install --upgrade scikit-learn

O modelo foi configurado com K = 6, assim a classificação de cada vinho será realizada considerando os 4 registros mais próximos na base de treinamento.

In [ ]:
modelo_classificador = KNeighborsClassifier(n_neighbors=6)
modelo_classificador.fit(x_train_escalonado, y_train)

In [ ]:
y_predito = modelo_classificador.predict(x_test_escalonado)

In [ ]:
y_predito

In [ ]:
from sklearn.metrics import accuracy_score

Verificação da acurácia do modelo:

In [ ]:
print(accuracy_score(y_test, y_predito))

Matriz de Confusão:

202 classificações corretas em 229 registros;

Acurácia de 88,21%;

17 de 32 vinhos de alta qualidade foram identificados corretamente (53,1%);

186 de 197 vinhos de baixa/média qualidade foram identificados corretamente (94,4%);

15 vinhos de alta qualidade foram classificados incorretamente como baixa/média qualidade;

11 vinhos de baixa/média qualidade foram classificados incorretamente como alta qualidade;

O modelo apresenta bom desempenho geral, mas ainda possui maior facilidade para identificar a classe de baixa/média qualidade;


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

matriz = confusion_matrix(
    y_test,
    y_predito,
    labels=modelo_classificador.classes_
)

grafico = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=modelo_classificador.classes_
)

grafico.plot()
plt.title("Matriz de confusão")
plt.show()

Relembrando a classificação da qualidade do vinho:

0 = Vinho de Baixa/Média Qualidade
1 = Vinho de Alta Qualidade

In [ ]:
dados[["quality", "classificacao"]].drop_duplicates().sort_values("quality")

Conclusão:

Foram realizados diferentes testes no modelo KNN, variando tanto o valor de K quanto as variáveis utilizadas na classificação. Após a correção de uma variável que estava duplicada, inicialmente foi testado o modelo com todas as 11 características físico-químicas, sendo que K = 8 apresentou o melhor resultado entre os valores avaliados, com 88,2% de acurácia e 202 classificações corretas em 229 registros.

Em seguida, foram realizados testes com um conjunto reduzido de variáveis consideradas mais relevantes. Com K = 5, o modelo obteve 87,8% de acurácia, apresentando desempenho inferior ao modelo com todas as variáveis.

Por fim, uma nova combinação de atributos foi avaliada utilizando K = 6, resultando no melhor desempenho entre os testes: 202 classificações corretas em 229 registros, com 88,21% de acurácia, além de identificar corretamente 12 dos 32 vinhos de alta qualidade.

Dessa forma, a configuração com K = 6 foi escolhida por apresentar o melhor equilíbrio entre a acurácia geral e a capacidade de identificar os vinhos de alta qualidade, que representam a classe minoritária da base.
